# 04｜移动目标与标准航线


> 运行方式：从上到下执行代码单元。控制类 Notebook **没有布尔安全开关**；执行控制单元就会把任务提交给 SDK。
> 需要仿真的课程请先在 FleetDock IDEA 中启动仿真，并确认 `drone1`、`drone2` 等设备在线。


![命令、Plan 与 Route](assets/plan-route.svg)


## 先生成航线

`Route` 提供圆形、网格、螺旋、走廊、测绘区域和自定义折线。生成航线不会控制设备；把航线传给 Plan 才会执行。


In [ ]:
from fleetdock import Route

circle = Route.circle(radius_m=6.0, segments=12, name="圆形巡检")
grid = Route.grid(width_m=12.0, height_m=8.0, spacing_m=4.0, name="网格覆盖")
spiral = Route.spiral(max_radius_m=10.0, spacing_m=3.0, turns=2.0, name="螺旋搜索")
corridor = Route.corridor(
    [{"x_m": 0, "y_m": 0}, {"x_m": 15, "y_m": 0}, {"x_m": 25, "y_m": 8}],
    width_m=6.0,
    name="走廊巡检",
)

for route in (circle, grid, spiral, corridor):
    print(route.name, route.pattern.value, len(route.waypoints), "个航点")


## 把航线交给 Plan

下面让 `drone1` 起飞后依次执行圆形和网格航线，再返航。速度、航高和超时都作为标准参数交给 SDK。


In [ ]:
from fleetdock_runtime import FleetDock

FleetDock.init()
drone = FleetDock.devices.drone1
print(drone.connect())

result = (
    FleetDock.core.plan(drone)
    .wait_until_ready(timeout_s=60.0)
    .unlock()
    .takeoff(altitude=30.0)
    .routes([circle, grid], altitude=30.0, speed=3.0, timeout_s=180.0)
    .return_to_launch()
    .run(step_timeout_s=180.0)
)
print(result)


In [ ]:
FleetDock.close()


## 重点

应用选择几何形状和执行参数，SDK 负责航点序列、任务执行和结果汇总。
